# EDA — Исследовательский анализ данных

Заполнение пропусков (медиана / мода), визуализации.

## 0. Импорты

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')
%matplotlib inline

## 1. Загрузка данных

👉 **Подставь путь к своему датасету.**

In [ ]:
# ===== ТВОЯ НАСТРОЙКА =====
DATASET_PATH = 'your_dataset.csv'
# ==========================

df = pd.read_csv(DATASET_PATH)
print(f'Shape: {df.shape}')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe()

## 2. Пропуски

In [ ]:
# Количество и доля пропусков по столбцам
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_df = pd.DataFrame({
    'missing_count': missing,
    'missing_%': missing_pct
}).sort_values('missing_%', ascending=False)

missing_df = missing_df[missing_df['missing_count'] > 0]

if missing_df.empty:
    print('✅ Пропусков нет!')
else:
    print(f'Столбцов с пропусками: {len(missing_df)}')
    display(missing_df)

In [ ]:
# Визуализация пропусков
if missing_df.empty:
    print('Пропусков нет — график не нужен.')
else:
    plt.figure(figsize=(8, max(3, len(missing_df) * 0.4)))
    missing_df['missing_%'].sort_values().plot.barh(
        color='salmon', edgecolor='black'
    )
    plt.xlabel('Пропуски, %')
    plt.title('Доля пропусков по столбцам')
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap пропусков (показывает паттерны)
if df.isnull().sum().sum() == 0:
    print('Пропусков нет — heatmap не нужен.')
else:
    plt.figure(figsize=(10, max(4, df.shape[1] * 0.25)))
    sns.heatmap(df.isnull(), cbar=True, yticklabels=False,
                cmap='YlOrRd', linewidths=0)
    plt.title('Пропуски (жёлтый = NaN)')
    plt.tight_layout()
    plt.show()

## 3. Заполнение пропусков

- **Числовые столбцы** → медиана
- **Категориальные столбцы** → мода

In [ ]:
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

print(f'Числовых столбцов: {len(num_cols)}')
print(f'Категориальных столбцов: {len(cat_cols)}')

In [ ]:
# Заполняем числовые — медианой
num_missing_before = df[num_cols].isnull().sum().sum()

for col in num_cols:
    if df[col].isnull().any():
        median_val = df[col].median()
        n_missing = df[col].isnull().sum()
        df[col] = df[col].fillna(median_val)
        print(f'{col}: заполнено {n_missing} пропусков медианой = {median_val:.4f}')

if num_missing_before == 0:
    print('Числовых пропусков не было.')

In [ ]:
# Заполняем категориальные — модой
cat_missing_before = df[cat_cols].isnull().sum().sum()

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode()[0]
        n_missing = df[col].isnull().sum()
        df[col] = df[col].fillna(mode_val)
        print(f'{col}: заполнено {n_missing} пропусков модой = "{mode_val}"')

if cat_missing_before == 0:
    print('Категориальных пропусков не было.')

In [ ]:
# Проверяем, что пропусков не осталось
remaining = df.isnull().sum().sum()
print(f'Осталось пропусков: {remaining}')

## 4. Распределения числовых признаков

👉 **Укажи список столбцов в `PLOT_COLS` (или оставь все числовые).**

In [ ]:
# ===== СТОЛБЦЫ ДЛЯ ГРАФИКОВ =====
PLOT_COLS = num_cols   # все числовые (измени список при необходимости)
# =================================

# Гистограммы с KDE
n = len(PLOT_COLS)
cols_per_row = 3
rows = (n + cols_per_row - 1) // cols_per_row

fig, axes = plt.subplots(rows, cols_per_row, figsize=(5 * cols_per_row, 4 * rows))
axes = np.array(axes).flatten()

for i, col in enumerate(PLOT_COLS):
    sns.histplot(df[col], kde=True, ax=axes[i], edgecolor='black')
    axes[i].set_title(col)

# Скрываем пустые оси
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 5. Boxplots — выбросы

In [ ]:
fig, axes = plt.subplots(rows, cols_per_row, figsize=(5 * cols_per_row, 4 * rows))
axes = np.array(axes).flatten()

for i, col in enumerate(PLOT_COLS):
    sns.boxplot(x=df[col], ax=axes[i], color='lightblue')
    axes[i].set_title(col)

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 6. Корреляционная матрица

In [ ]:
corr = df[PLOT_COLS].corr()

plt.figure(figsize=(max(8, len(PLOT_COLS) * 0.7),
              max(6, len(PLOT_COLS) * 0.6)))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5)
plt.title('Корреляционная матрица')
plt.tight_layout()
plt.show()

## 7. Pairplot (первые N признаков)

Если признаков много — pairplot тяжёлый. Ограничь `PAIR_COLS`.

In [ ]:
# ===== ПРИЗНАКИ ДЛЯ PAIRPLOT =====
PAIR_COLS = PLOT_COLS[:5]   # первые 5 (измени)
# =================================

if len(PAIR_COLS) < 2:
    print('Нужно минимум 2 признака для pairplot.')
else:
    sns.pairplot(df[PAIR_COLS], diag_kind='kde', plot_kws={'alpha': 0.3, 's': 12})
    plt.show()

## 8. Категориальные признаки — распределения

In [ ]:
if len(cat_cols) == 0:
    print('Категориальных столбцов нет.')
else:
    n_cat = len(cat_cols)
    fig, axes = plt.subplots(1, n_cat, figsize=(5 * n_cat, 4))
    if n_cat == 1:
        axes = [axes]
    for ax, col in zip(axes, cat_cols):
        vc = df[col].value_counts()
        # Если уникальных значений много — показываем топ-10
        if len(vc) > 10:
            vc = vc.head(10)
            ax.set_title(f'{col} (top 10)')
        else:
            ax.set_title(col)
        vc.plot.bar(ax=ax, edgecolor='black', color='steelblue')
        ax.tick_params(axis='x', rotation=30)
    plt.tight_layout()
    plt.show()

## 9. Scatter: числовой признак vs таргет

👉 **Укажи имя таргета.** Можно пропустить, если таргет ещё не выделен.

In [ ]:
# ===== ИМЯ ТАЙРЖЕТА =====
TARGET_COL = None   # например: 'price', 'label', 'target'
# ========================

if TARGET_COL is None or TARGET_COL not in df.columns:
    print('TARGET_COL не задан или не найден — пропускаем scatter vs target.')
else:
    scatter_cols = [c for c in PLOT_COLS if c != TARGET_COL][:6]
    if len(scatter_cols) == 0:
        print('Нет числовых признаков для scatter.')
    else:
        fig, axes = plt.subplots(1, len(scatter_cols),
                                figsize=(5 * len(scatter_cols), 4))
        if len(scatter_cols) == 1:
            axes = [axes]
        for ax, col in zip(axes, scatter_cols):
            ax.scatter(df[col], df[TARGET_COL], alpha=0.3, s=10)
            ax.set_xlabel(col)
            ax.set_ylabel(TARGET_COL)
        plt.tight_layout()
        plt.show()

## 10. Итоговая таблица пропусков (ДО / ПОСЛЕ)

In [ ]:
summary = pd.DataFrame({
    'dtype': df.dtypes,
    'non-null': df.notnull().sum(),
    'unique': df.nunique(),
    'mean/median': [round(df[c].median(), 4) if c in num_cols else '-' for c in df.columns],
    'mode': [df[c].mode()[0] if df[c].mode().any() else '-' for c in df.columns],
})
summary

## 11. Сохранение очищенного датасета

In [ ]:
OUTPUT_PATH = 'dataset_clean.csv'
df.to_csv(OUTPUT_PATH, index=False)
print(f'Сохранено в {OUTPUT_PATH}  —  {df.shape[0]} строк, {df.shape[1]} столбцов')